In [6]:
from pymilvus import (
    connections, FieldSchema, CollectionSchema, DataType, Collection, utility
)
import numpy as np
import pandas as pd

from nltk.corpus import stopwords
import spacy
import nltk
import json
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

connections.connect("default", host="localhost", port="19530")
# nltk.download('stopwords')
# spacy.cli.download("en_core_web_sm")

In [7]:
# fields = [
#     FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=384),
#     FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=65535, is_primary=True),
#     FieldSchema(name="year", dtype=DataType.INT64),
#     FieldSchema(name="month", dtype=DataType.INT64),
#     FieldSchema(name="day", dtype=DataType.INT64),
#     FieldSchema(
#         name="words",
#         dtype=DataType.ARRAY,
#         element_type=DataType.VARCHAR,
#         max_length=400,
#         max_capacity=4096
#     ),
#     FieldSchema(
#         name="entity_names",
#         dtype=DataType.ARRAY,
#         element_type=DataType.VARCHAR,
#         max_length=300,
#         max_capacity=600
#     ),
#     FieldSchema(
#         name="entity_ids",
#         dtype=DataType.ARRAY,
#         element_type=DataType.VARCHAR,
#         max_length=300,
#         max_capacity=600
#     ),
# ]


# schema = CollectionSchema(fields=fields, description="Collection of Russian Speeches")

# collection_name = "russian_speeches"
# if collection_name in utility.list_collections():
#     utility.drop_collection(collection_name)

# collection = Collection(name=collection_name, schema=schema)
# collection.create_index(
#     field_name="embedding",
#     index_params={
#         "index_type": "IVF_FLAT",
#         "metric_type": "COSINE",
#         "params": {"nlist": 1024}
#     }
# )
# print(f"Collection `{collection_name}` created successfully.")

In [8]:
collection = Collection("russian_speeches")
collection.load()

In [45]:
with open("data/putin_complete.json", "r") as f:
    speeches = json.load(f)

stop_words = set(stopwords.words('english'))
punctuation = [".", ",", "?", "!", ":", "`", "'", "(", ")", "[", "]", "/", '’', "-", "’s", "\"", ";", "i", " ", "–", "%", "*", "...", "…"]
lemmatize = spacy.load("en_core_web_sm")
model = SentenceTransformer("all-MiniLM-L6-v2")

In [46]:
data = [[],[],[],[],[],[],[],[]]

In [ ]:
for speech in tqdm((speeches), "Populating collection..."):
    
    text = speech["transcript_filtered"]
    if len(text) > 35000:
        # print(f"Text skipped with: {len(text)} signs.")
        continue
    
    doc = lemmatize(text)

    splitted_date = speech["date"].split("-")
    year = int(splitted_date[0])
    month = int(splitted_date[1])
    day = int(splitted_date[2].split("T")[0])

    words = [
        w.lemma_.lower() for w in doc if not (w.lemma_ in stop_words or w.lemma_.lower() in punctuation or " " in w.lemma_)
    ]

    entity_names = []
    entity_ids = []

    for ent in doc.ents:
        entity_names.append(ent.text)
        entity_ids.append(f"{ent.start},{ent.end}")
    # data = [[],[],[],[],[],[],[],[]]
    data[0].append(model.encode(text))
    data[1].append(text)
    data[2].append(year)
    data[3].append(month)
    data[4].append(day)
    data[5].append(words)
    data[6].append(entity_names)
    data[7].append(entity_ids)


collection.insert(data)

In [17]:
results = collection.query(
    expr="year == 2020",
    output_fields=["year", "month", "day", "words"]
)

results[0]

{'year': 2020,
 'month': 2,
 'day': 26,
 'words': ['never', 'call', 'judge', 'meet', 'judge', 'time', 'time', 'mean', 'former', 'university', 'classmate', 'many', 'judge', 'among', 'currently', 'office', 'however', 'meet', 'classmate', 'capacity', 'judge', 'class-', 'uni', 'mate', 'well', 'let', 'us', 'say', 'invent', 'yes', 'hear', 'nothing', 'good', 'true', 'yes', 'happen', 'good', 'good', 'look', 'want', 'say', 'people', 'getting', 'involve', 'make', 'difference', 'today', 'russia', 'good', 'currently', 'think', 'situation', 'evolve', 'simple', 'way', 'law', 'enforcement', 'agency', 'look', 'matter', 'fire', 'detain', 'want', 'beat', 'confession', 'people', 'obtain', 'due', 'course', 'accordance', 'law', 'latter', 'seem', 'well', 'option', 'take', 'time', 'rush', 'haste', 'appear', 'president', 'act', 'guarantor', 'constitution', 'right', 'must', 'respond', 'sort', 'issue', 'see', 'happen', 'real', 'life', 'know', 'prison', 'population', 'early', '2000s', 'actually', 'halve', 'yes',